# 個人別の指数学習率のばらつきは経路条件で異なるか

各参加者のpractice 1〜400試行の指数回帰で推定した学習率 **c（試行⁻¹）** を使う。対象はcore MTと1/TPe。C1（ベースライン）、C2（長距離）、C3（幅狭）の各10名が独立な3群を構成する。条件平均曲線の係数は検定に使わない。前回の個人別係数を丸めずに用い、全員を含める。

条件別に標本分散（分母 n−1、試行⁻²）、標準偏差（試行⁻¹）、四分位範囲を求める。主な比較は、各値と群中央値との差の絶対値を用いたBrown–Forsythe検定（中央値版Levene検定、F近似）。3群間の等分散の帰無仮説を各指標について調べ、2検定をHolm補正する。補足の3条件対 × 2指標も同じ方法で検定し、6件で別にHolm補正する。有意水準は5%。

この分析の対象は各人の曲線から得た推定値のばらつきで、係数の推定誤差も含む。小標本で群間の分布形状も異なり得るため、検定は近似である。[NISTによる検定の定義](https://www.itl.nist.gov/div898/handbook/eda/section3/eda35a.htm)

In [1]:
import csv
import math
import os
from collections import Counter
from html import escape
from itertools import combinations
from pathlib import Path
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','VECLIB_MAXIMUM_THREADS','MKL_NUM_THREADS'):
    os.environ[key] = '1'
import numpy as np
from scipy import stats
from IPython.display import HTML, display

root = next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'analysis.py').is_file() and (p/'inputCSV').is_dir())
base = root/'output/learning_report'
source = base/'individual_exponential_rates_20261128_212205/individual_rates.csv'
out = base/'individual_rate_variances_20261128_214313'
conditions = ('C1','C2','C3')
metrics = ('core MT','1/TPe')
with source.open(encoding='utf-8-sig',newline='') as f:
    people = list(csv.DictReader(f))
assert len({r['参加者ID'] for r in people}) == len(people) == 30
assert Counter(r['条件'] for r in people) == {c:10 for c in conditions}
members = {c:[r for r in people if r['条件']==c] for c in conditions}
groups = {m:{c:np.array([float(r[m+' c']) for r in persons]) for c,persons in members.items()} for m in metrics}
assert all(np.isfinite(x).all() and (x>0).all() for groups_m in groups.values() for x in groups_m.values())

def save_csv(name, rows):
    with (out/name).open('w',encoding='utf-8-sig',newline='') as f:
        writer = csv.DictWriter(f,fieldnames=rows[0])
        writer.writeheader()
        writer.writerows(rows)

def table(headers, rows):
    header = ''.join('<th>'+escape(str(v))+'</th>' for v in headers)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(v))+'</td>' for v in row)+'</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))

## 条件別の標本分散と標準偏差

「最大寄与」は、条件内の偏差平方和のうち、平均との差が最も大きい1名の偏差平方が占める割合。同じ割合が群の標本分散の和の各項についても成り立つ。四分位範囲はQ3−Q1（NumPyの線形補間）で求めた。

In [2]:
summary = []
for m in metrics:
    for c,x in groups[m].items():
        squares = (x-x.mean())**2
        idx = int(squares.argmax())
        person = members[c][idx]
        q1,q3 = np.quantile(x,[.25,.75])
        summary.append({'指標':m,'条件':c,'人数':len(x),'標本分散':x.var(ddof=1),'標準偏差':x.std(ddof=1),
                        '四分位範囲':q3-q1,'偏差平方の最大寄与ID':person['参加者ID'],'最大寄与者のc':x[idx],
                        '偏差平方和に占める割合_pct':100*squares[idx]/squares.sum(),
                        '最大寄与者の注記':person[m+' 解釈上の注記']})
save_csv('condition_variances.csv',summary)
table(['指標','条件','人数','標本分散','標準偏差','四分位範囲','最大寄与ID','割合'],
      [[r['指標'],r['条件'],r['人数'],f"{r['標本分散']:.6f}",f"{r['標準偏差']:.5f}",
        f"{r['四分位範囲']:.5f}",r['偏差平方の最大寄与ID'],f"{r['偏差平方和に占める割合_pct']:.1f}%"] for r in summary])

指標,条件,人数,標本分散,標準偏差,四分位範囲,最大寄与ID,割合
core MT,C1,10,0.010007,0.10004,0.10361,028,54.9%
core MT,C2,10,0.003803,0.06167,0.01654,006,73.1%
core MT,C3,10,1.214848,1.10220,0.02085,014,89.7%
1/TPe,C1,10,0.043331,0.20816,0.11508,028,73.2%
1/TPe,C2,10,2.442477,1.56284,0.14857,003,89.0%
1/TPe,C3,10,2.464291,1.56981,0.08323,014,89.9%


## 等分散の検定

3群の全体比較と条件対ごとの補足比較を示す。各行の標本は参加者であり、各人の400試行を独立観測として数えない。

In [3]:
def compare(metric, names, family):
    samples = [groups[metric][c] for c in names]
    result = stats.levene(*samples,center='median')
    return {'指標':metric,'比較':'–'.join(names),'F':float(result.statistic),'分子自由度':len(names)-1,
            '分母自由度':sum(map(len,samples))-len(names),'未補正p':float(result.pvalue),'補正群':family}

def correct(rows):
    maximum = 0.
    for rank,r in enumerate(sorted(rows,key=lambda x:x['未補正p'])):
        maximum = max(maximum,min(1.,(len(rows)-rank)*r['未補正p']))
        r['Holm補正p'] = maximum
        r['判定_5%'] = '有意' if maximum < .05 else '非有意'
    return rows

omnibus = correct([compare(m,conditions,'全体2検定') for m in metrics])
pairs = correct([compare(m,pair,'条件対6検定') for m in metrics for pair in combinations(conditions,2)])
tests = omnibus+pairs
save_csv('tests.csv',tests)
table(['指標','比較','F','df1','df2','未補正p','Holm補正p','判定'],
      [[r['指標'],r['比較'],f"{r['F']:.4f}",r['分子自由度'],r['分母自由度'],
        f"{r['未補正p']:.5f}",f"{r['Holm補正p']:.5f}",r['判定_5%']] for r in tests])

指標,比較,F,df1,df2,未補正p,Holm補正p,判定
core MT,C1–C2–C3,0.8967,2,27,0.41972,0.83943,非有意
1/TPe,C1–C2–C3,0.3944,2,27,0.67791,0.83943,非有意
core MT,C1–C2,0.9162,1,18,0.35114,1.00000,非有意
core MT,C1–C3,0.8047,1,18,0.38153,1.00000,非有意
core MT,C2–C3,0.9889,1,18,0.33320,1.00000,非有意
1/TPe,C1–C2,0.8555,1,18,0.36723,1.00000,非有意
1/TPe,C1–C3,0.7029,1,18,0.41280,1.00000,非有意
1/TPe,C2–C3,0.0036,1,18,0.95309,1.00000,非有意


## 推定上の注意と結論

core MTの014は指数曲線がほぼ平坦でR²が0であり、学習率の値をデータから特定しにくい。1/TPeの003と014は推定上限5に達している。これらの参加者を含む条件の分散にどの程度影響しているかを、偏差平方和に占める割合で確認する。全員を含めた検定の有意・非有意は上表に示した。

In [4]:
for m,pid in [('core MT','014'),('1/TPe','003'),('1/TPe','014')]:
    p = next(r for r in people if r['参加者ID']==pid)
    x = groups[m][p['条件']]
    share = (float(p[m+' c'])-x.mean())**2 / np.sum((x-x.mean())**2)
    print(f"{m}・{p['条件']}の{pid}　偏差平方和の{100*share:.1f}%（{p[m+' 解釈上の注記']}）")
for r in omnibus:
    print(f"{r['指標']}　F({r['分子自由度']}, {r['分母自由度']}) = {r['F']:.3f}、Holm補正p = {r['Holm補正p']:.4f}（{r['判定_5%']}）。")

core MT・C3の014　偏差平方和の89.7%（振幅ほぼ0・cを解釈不能）
1/TPe・C2の003　偏差平方和の89.0%（cが上限5）
1/TPe・C3の014　偏差平方和の89.9%（cが上限5）
core MT　F(2, 27) = 0.897、Holm補正p = 0.8394（非有意）。
1/TPe　F(2, 27) = 0.394、Holm補正p = 0.8394（非有意）。
